In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("data_transformations").getOrCreate()
print("Spark session ready")

Spark session ready


In [2]:
sensor_logs = """log_id,device_id,room_id,timestamp,energy_kwh
1,2,1,2026-05-01 09:00:00,1.2000
2,2,1,2026-05-01 13:00:00,1.1000
3,2,1,2026-05-01 18:00:00,1.3500
4,1,1,2026-05-01 20:00:00,0.1500
5,1,1,2026-05-02 21:00:00,0.1800
6,3,2,2026-05-01 10:00:00,0.0500
7,3,2,2026-05-01 15:00:00,0.0600
8,3,2,2026-05-02 11:00:00,0.0550
9,5,3,2026-05-01 08:00:00,0.4000
10,5,3,2026-05-01 12:00:00,0.4200
11,5,3,2026-05-02 08:00:00,0.4100
12,5,3,2026-05-02 20:00:00,0.4500
13,7,4,2026-05-01 09:30:00,0.0800
14,7,4,2026-05-01 17:30:00,0.0900
15,7,4,2026-05-02 09:30:00,0.0850
16,8,4,2026-05-01 00:00:00,0.0200
17,8,4,2026-05-01 06:00:00,0.0220
18,8,4,2026-05-02 00:00:00,0.0210
19,2,1,2026-06-01 10:00:00,1.5000
20,2,1,2026-06-01 15:00:00,1.4500
21,5,3,2026-06-06 08:00:00,0.4800
22,5,3,2026-06-06 20:00:00,0.5000
23,2,1,2026-07-01 12:00:00,2.5000
24,5,3,2026-07-01 08:00:00,0.6000
25,3,2,2026-07-02 09:00:00,0.0700
26,3,2,2026-07-02 14:00:00,0.0750
27,7,4,2026-07-03 10:00:00,0.1000
28,7,4,2026-07-03 18:00:00,0.1100
29,8,4,2026-07-04 00:00:00,0.0300
30,8,4,2026-07-04 06:00:00,0.0320
"""

with open("energy_logs.csv", "w") as f:
    f.write(sensor_logs)

logs_df= spark.read.csv("energy_logs.csv", header= True, inferSchema=True)
logs_df.show()

+------+---------+-------+-------------------+----------+
|log_id|device_id|room_id|          timestamp|energy_kwh|
+------+---------+-------+-------------------+----------+
|     1|        2|      1|2026-05-01 09:00:00|       1.2|
|     2|        2|      1|2026-05-01 13:00:00|       1.1|
|     3|        2|      1|2026-05-01 18:00:00|      1.35|
|     4|        1|      1|2026-05-01 20:00:00|      0.15|
|     5|        1|      1|2026-05-02 21:00:00|      0.18|
|     6|        3|      2|2026-05-01 10:00:00|      0.05|
|     7|        3|      2|2026-05-01 15:00:00|      0.06|
|     8|        3|      2|2026-05-02 11:00:00|     0.055|
|     9|        5|      3|2026-05-01 08:00:00|       0.4|
|    10|        5|      3|2026-05-01 12:00:00|      0.42|
|    11|        5|      3|2026-05-02 08:00:00|      0.41|
|    12|        5|      3|2026-05-02 20:00:00|      0.45|
|    13|        7|      4|2026-05-01 09:30:00|      0.08|
|    14|        7|      4|2026-05-01 17:30:00|      0.09|
|    15|      

In [4]:
from pyspark.sql.functions import col, to_timestamp, month, year, sum, avg, when
logs_df= logs_df.withColumn("timestamp", to_timestamp(col("timestamp")))

logs_df =logs_df.filter(col("timestamp").isNotNull())
logs_df= logs_df.withColumn("month", month(col("timestamp")))\
                .withColumn("year", year(col("timestamp")))
logs_df.show()

+------+---------+-------+-------------------+----------+-----+----+
|log_id|device_id|room_id|          timestamp|energy_kwh|month|year|
+------+---------+-------+-------------------+----------+-----+----+
|     1|        2|      1|2026-05-01 09:00:00|       1.2|    5|2026|
|     2|        2|      1|2026-05-01 13:00:00|       1.1|    5|2026|
|     3|        2|      1|2026-05-01 18:00:00|      1.35|    5|2026|
|     4|        1|      1|2026-05-01 20:00:00|      0.15|    5|2026|
|     5|        1|      1|2026-05-02 21:00:00|      0.18|    5|2026|
|     6|        3|      2|2026-05-01 10:00:00|      0.05|    5|2026|
|     7|        3|      2|2026-05-01 15:00:00|      0.06|    5|2026|
|     8|        3|      2|2026-05-02 11:00:00|     0.055|    5|2026|
|     9|        5|      3|2026-05-01 08:00:00|       0.4|    5|2026|
|    10|        5|      3|2026-05-01 12:00:00|      0.42|    5|2026|
|    11|        5|      3|2026-05-02 08:00:00|      0.41|    5|2026|
|    12|        5|      3|2026-05-

In [5]:
monthly_usage_df= logs_df.groupby("month","room_id").agg(sum("energy_kwh").alias("monthly_energy_kwh"))

monthly_usage_df.show()

+-----+-------+-------------------+
|month|room_id| monthly_energy_kwh|
+-----+-------+-------------------+
|    6|      1|               2.95|
|    7|      4|              0.272|
|    7|      1|                2.5|
|    7|      2|0.14500000000000002|
|    5|      4|0.31800000000000006|
|    7|      3|                0.6|
|    6|      3|               0.98|
|    5|      2|              0.165|
|    5|      1|               3.98|
|    5|      3|               1.68|
+-----+-------+-------------------+



In [8]:
device_monthly_usage = logs_df.groupby("device_id", "room_id", "month")\
                       .agg(sum("energy_kwh").alias("monthly_device_energy_kwh"))
device_monthly_usage.show()

+---------+-------+-----+-------------------------+
|device_id|room_id|month|monthly_device_energy_kwh|
+---------+-------+-----+-------------------------+
|        2|      1|    7|                      2.5|
|        5|      3|    7|                      0.6|
|        8|      4|    7|                    0.062|
|        7|      4|    7|      0.21000000000000002|
|        5|      3|    6|                     0.98|
|        8|      4|    5|                    0.063|
|        5|      3|    5|                     1.68|
|        3|      2|    5|                    0.165|
|        2|      1|    5|                     3.65|
|        7|      4|    5|                    0.255|
|        2|      1|    6|                     2.95|
|        3|      2|    7|      0.14500000000000002|
|        1|      1|    5|      0.32999999999999996|
+---------+-------+-----+-------------------------+



In [9]:
device_avg = device_monthly_usage.groupby("device_id")\
             .agg(avg("monthly_device_energy_kwh").alias("avg_monthly_usage"))

spike_df= device_monthly_usage.join(device_avg, on="device_id")

spike_df = spike_df.withColumn("spike_alert", when( col("monthly_device_energy_kwh") > col("avg_monthly_usage") * 1.5, "spike")\
                               .otherwise("Normal"))
spike_df.show()

+---------+-------+-----+-------------------------+-------------------+-----------+
|device_id|room_id|month|monthly_device_energy_kwh|  avg_monthly_usage|spike_alert|
+---------+-------+-----+-------------------------+-------------------+-----------+
|        1|      1|    5|      0.32999999999999996|0.32999999999999996|     Normal|
|        3|      2|    7|      0.14500000000000002|0.15500000000000003|     Normal|
|        3|      2|    5|                    0.165|0.15500000000000003|     Normal|
|        5|      3|    5|                     1.68| 1.0866666666666667|      spike|
|        5|      3|    6|                     0.98| 1.0866666666666667|     Normal|
|        5|      3|    7|                      0.6| 1.0866666666666667|     Normal|
|        8|      4|    5|                    0.063|             0.0625|     Normal|
|        8|      4|    7|                    0.062|             0.0625|     Normal|
|        7|      4|    5|                    0.255|             0.2325|     

In [10]:
spikes_only = spike_df.filter(col("spike_alert") == "spike")
spikes_only.show()

+---------+-------+-----+-------------------------+------------------+-----------+
|device_id|room_id|month|monthly_device_energy_kwh| avg_monthly_usage|spike_alert|
+---------+-------+-----+-------------------------+------------------+-----------+
|        5|      3|    5|                     1.68|1.0866666666666667|      spike|
+---------+-------+-----+-------------------------+------------------+-----------+



In [11]:
#top_energy_consuming
top_devices = logs_df.groupby("device_id").agg(sum("energy_kwh").alias("total_energy_kwh"))\
                     .orderBy(col("total_energy_kwh").desc())
top_devices.show()


+---------+-------------------+
|device_id|   total_energy_kwh|
+---------+-------------------+
|        2|  9.100000000000001|
|        5| 3.2600000000000002|
|        7|0.46499999999999997|
|        1|0.32999999999999996|
|        3|               0.31|
|        8|              0.125|
+---------+-------------------+

